# Train and test the design PDF's pipeline — 5-fold CV

Runs `gatetrain`, which implements the slides rather than a substitute for them.

| PDF | |
|---|---|
| **p2** | `L_NCE` — InfoNCE (τ = 0.1) between `z_M` and `ẑ_M`, predicted from the two frozen frame features |
| **p3** | `L_sim` — MSE regression of `[S_frame, S_gaze]` from `z_M` |
| **p9** | `L = L_NCE + L_similarity` |
| **p4** | `FilterFrameForVLM`, at the **same thresholds that created the labels** |

```
frame t-1 ──► [DINOv2 CLS, frozen] ──► z_I0 ─┐
                                              ├─► h ──► ẑ_M ─┐
frame t   ──► [DINOv2 CLS, frozen] ──► z_IT ─┘               │ L_NCE
                                                             │
gaze rates (3,9) ──► f_M ──► z_M ────────────────────────────┘
                              │
                              └──► head ──► [S_frame, S_gaze] ──► L_sim
```

Only `f_M` + head are deployed. `h` exists to build the Loss-1 target and is thrown away
at inference, where **no frame is encoded at all**.

## What is different from the earlier gate notebook

That one trained cross-entropy on hard `quad` labels — a reasonable baseline, but not the
PDF's method, and not distillation or self-supervision. This one predicts the two
**continuous** similarities and lets `FilterFrameForVLM` do the thresholding, so the
two-score design is the mechanism rather than something baked into the labels.

## Run order

1. Clone, mount, check the folds and `thresholds.json` agree
2. **Train** — joint `L_NCE + L_sim` across 5 folds
3. **`--sim_only` ablation** — what is the contrastive term actually worth?
4. **Shuffle control** — must fail
5. **Test**, once, through `FilterFrameForVLM`

## Cost

Frame features are read from Drive **once** and cached locally (~20 MB for 13k frames),
so only the first cell pays that. Training is a couple of minutes for all five folds.
Set the runtime to a **T4**, though the model is small enough that it is not the limit.

## 1 — Clone, mount, check the GPU

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

import os, json, time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"
FOLDS     = os.path.join(DRIVE_DIR, "folds")
THRESH    = os.path.join(DRIVE_DIR, "thresholds.json")
RUNS      = "/content/runs/joint"

print("\ntorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU  ", torch.cuda.get_device_name(0))
else:
    print("no GPU -- Runtime > Change runtime type > T4. It will still run on CPU.")

## 2 — Check the folds and the thresholds belong together

`thresholds.json` holds the τ_frame / τ_gaze that produced the `quad` labels. The check
below applies them to the **true** similarities and confirms they reproduce the stored
labels.

This is not ceremony. If the labelling notebook were re-run with different percentiles and
the folds not rebuilt, every number downstream would be measured against the wrong
boundary — and nothing in the output would look wrong.

In [ ]:
need = [f"train_val_fold{k}.csv" for k in range(1, 6)] + ["test.csv"]
missing = [f for f in need if not os.path.exists(os.path.join(FOLDS, f))]
assert not missing, f"missing in {FOLDS}: {missing}\nRun colab_label_quadrants.ipynb first."
assert os.path.exists(THRESH), f"missing {THRESH}"

th = json.load(open(THRESH))
TAU_F, TAU_G = th["tau_frame"], th["tau_gaze"]
print(f"tau_frame {TAU_F:.4f}   tau_gaze {TAU_G:.4f}")
print(f"   percentiles {th.get('frame_pct')}/{th.get('gaze_pct')}   "
      f"per_video={th.get('per_video')}   computed on {th.get('computed_on')}")

for f in need:
    d = pd.read_csv(os.path.join(FOLDS, f),
                    usecols=["sequence", "frame_similarity", "gaze_patch_token_sim", "quad"])
    q = 2*(d.frame_similarity > TAU_F).astype(int) + (d.gaze_patch_token_sim > TAU_G).astype(int)
    agree = float((q == d["quad"]).mean())
    print(f"   {f:<24} {len(d):6,} rows  {d.sequence.nunique():3d} videos   "
          f"tau reproduces quad on {100*agree:6.2f}%   [{'PASS' if agree > 0.999 else 'FAIL'}]")

te = pd.read_csv(os.path.join(FOLDS, "test.csv"), usecols=["sequence", "quad"])
f1 = pd.read_csv(os.path.join(FOLDS, "train_val_fold1.csv"), usecols=["sequence", "quad"])
print(f"\nvideos shared between test and the folds: {len(set(te.sequence) & set(f1.sequence))}"
      f"   [{'PASS' if not (set(te.sequence) & set(f1.sequence)) else 'FAIL'}]")

NAMES = {0:"TRANSITION",1:"PURSUIT",2:"REFIXATION",3:"STABLE"}
print("\nquadrant balance:")
display(pd.DataFrame({
    "folds": f1["quad"].value_counts(normalize=True).mul(100).round(1),
    "test":  te["quad"].value_counts(normalize=True).mul(100).round(1),
}).rename(index=NAMES).fillna(0))
print("REFIXATION is the class the two-threshold design exists for. If it is tiny here,")
print("its recall later will be noisy no matter how good the model is.")

## 3 — Train: `L = L_NCE + L_sim`

The first run builds the frame-feature cache — one `.npz` per unique frame off Drive,
collapsed into a `(n_frames, 384)` matrix and saved locally. Later runs reuse it.

Watch **both loss terms**. `L_NCE` starts near ln(batch) ≈ 6.2 and `L_sim` near 1.0, so a
plain sum (which is what p9 specifies) lets the contrastive term dominate early. If `sim`
stops falling while `nce` keeps improving, lower `--nce_weight`.

In [ ]:
EPOCHS, PATIENCE, BATCH, LR = 200, 25, 512, 3e-4
RATES = "gaze_rates_window"        # or gaze_vec3d_rates_window (sphere-exact)

t0 = time.time()
!python -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --feat_cache /content/feat_cache.npz

print(f"\ntotal {(time.time()-t0)/60:.1f} min")

### Curves and the fold spread

In [ ]:
res = json.load(open(os.path.join(RUNS, "cv_results.json")))
H = res["history"]

fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
for k, h in H.items():
    ep = [r["epoch"] for r in h]
    ax[0].plot(ep, [r["tr_sim"] for r in h], lw=1.2, label=k)
    if "tr_nce" in h[0]:
        ax[1].plot(ep, [r.get("tr_nce", np.nan) for r in h], lw=1.2, label=k)
    ax[2].plot(ep, [r["auc_gate"] for r in h], lw=1.4, label=k)
    ax[3].plot(ep, [r["r2_mean"] for r in h], lw=1.4, label=k)

ax[0].set_title("L_sim  (PDF p3)"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)
ax[1].set_title("L_NCE  (PDF p2)"); ax[1].set_xlabel("epoch")
ax[1].axhline(np.log(512), ls="--", c="k", lw=1, label="ln(batch)"); ax[1].legend(fontsize=7)
ax[2].axhline(0.5, ls="--", c="k", lw=1)
ax[2].set_title("val AUC_gate", fontweight="bold"); ax[2].set_xlabel("epoch")
ax[3].axhline(0.0, ls="--", c="k", lw=1, label="predict-the-mean")
ax[3].set_title("val R2", fontweight="bold"); ax[3].set_xlabel("epoch"); ax[3].legend(fontsize=7)
plt.tight_layout(); plt.show()

tab = pd.DataFrame([f["last"] for f in res["folds"]])
tab.index = [f"fold {i+1}" for i in range(len(tab))]
display(tab[["auc_gate","auc_frame","auc_gaze","r2_mean","quad_acc","recall_q2"]].round(4))

auc = tab.auc_gate.to_numpy(); r2 = tab.r2_mean.to_numpy()
print(f"AUC_gate  {auc.mean():.4f} +/- {auc.std():.4f}")
print(f"R2        {r2.mean():+.3f} +/- {r2.std():.3f}   <- read this first; it has a baseline")
print(f"REFIX rec {tab.recall_q2.mean():.3f}")
if r2.mean() <= 0:
    print("\n  R2 <= 0: no better than predicting the training mean on held-out videos.")
    print("  AUC can still look respectable when that is true.")

## 4 — Ablation: is `L_NCE` earning its place?

`--sim_only` trains Loss 2 alone. Same data, same head, no contrastive term and no frame
features at all.

If the joint run is not better than this, then `L_NCE` — and with it the entire p2
machinery, the change predictor, and the need to load frame features — is costing
complexity for nothing. That is worth knowing before it goes in a paper.

In [ ]:
RUNS_SIM = "/content/runs/sim_only"

!python -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS_SIM}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --sim_only 2>&1 | tail -14

sim = json.load(open(os.path.join(RUNS_SIM, "cv_results.json")))
j_auc = np.array([f["last"]["auc_gate"] for f in res["folds"]])
s_auc = np.array([f["last"]["auc_gate"] for f in sim["folds"]])
j_r2  = np.array([f["last"]["r2_mean"] for f in res["folds"]])
s_r2  = np.array([f["last"]["r2_mean"] for f in sim["folds"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a_, (j, s, t) in zip(ax, [(j_auc, s_auc, "AUC_gate"), (j_r2, s_r2, "R2")]):
    a_.bar(np.arange(5)-.19, j, .38, label=f"joint  {j.mean():.3f}")
    a_.bar(np.arange(5)+.19, s, .38, label=f"sim_only  {s.mean():.3f}", color="tab:orange")
    a_.set_xticks(range(5)); a_.set_xticklabels([f"f{i+1}" for i in range(5)])
    a_.set_title(t); a_.legend()
ax[0].axhline(0.5, ls="--", c="k", lw=1); ax[1].axhline(0, ls="--", c="k", lw=1)
plt.tight_layout(); plt.show()

d = j_auc.mean() - s_auc.mean()
print(f"joint - sim_only:  AUC {d:+.4f}   R2 {j_r2.mean()-s_r2.mean():+.4f}")
if d < 0.01:
    print("\n  L_NCE is not buying anything measurable here. The honest options are to")
    print("  report it as an ablation that did not help, or to tune --nce_weight before")
    print("  concluding -- a plain sum may simply be swamping L_sim.")
else:
    print("\n  The contrastive term helps. That is the p2/p9 claim, supported.")

## 5 — Shuffle control

Each row's frames are paired with **another row's** gaze, destroying the correspondence
both losses depend on. **R² must stay ≤ 0 and AUC near 0.5.** If it does not, the setup
leaks and nothing above is credible.

In [ ]:
RUNS_CTRL = "/content/runs/joint_control"

!python -m gatetrain.train \
    --folds_dir "{FOLDS}" --out_dir "{RUNS_CTRL}" --thresholds "{THRESH}" \
    --rates_col {RATES} --epochs {EPOCHS} --patience {PATIENCE} \
    --batch_size {BATCH} --lr {LR} --shuffle_control 2>&1 | tail -12

ctrl = json.load(open(os.path.join(RUNS_CTRL, "cv_results.json")))
c_auc = np.array([f["last"]["auc_gate"] for f in ctrl["folds"]])
c_r2  = np.array([f["last"]["r2_mean"] for f in ctrl["folds"]])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(np.arange(5)-.19, j_auc, .38, label=f"real  {j_auc.mean():.3f}")
ax[0].bar(np.arange(5)+.19, c_auc, .38, label=f"shuffled  {c_auc.mean():.3f}", color="tab:red")
ax[0].axhline(0.5, ls="--", c="k", lw=1); ax[0].set_title("AUC_gate"); ax[0].legend()
ax[1].bar(np.arange(5)-.19, j_r2, .38, label=f"real  {j_r2.mean():+.3f}")
ax[1].bar(np.arange(5)+.19, c_r2, .38, label=f"shuffled  {c_r2.mean():+.3f}", color="tab:red")
ax[1].axhline(0, ls="--", c="k", lw=1); ax[1].set_title("R2"); ax[1].legend()
for a_ in ax:
    a_.set_xticks(range(5)); a_.set_xticklabels([f"f{i+1}" for i in range(5)])
plt.tight_layout(); plt.show()

print(f"real     AUC {j_auc.mean():.4f}   R2 {j_r2.mean():+.3f}")
print(f"shuffled AUC {c_auc.mean():.4f}   R2 {c_r2.mean():+.3f}")
if c_r2.mean() > 0.05:
    print("\n  !! THE CONTROL LEARNED SOMETHING. Suspect the split or the normalisation")
    print("     before reporting anything above.")
elif j_r2.mean() - c_r2.mean() < 0.05:
    print("\n  Real and shuffled are indistinguishable: gaze is not predicting these")
    print("  similarities at this scale. A clean negative, not a bug.")
else:
    print("\n  Control fails as it must, and the real run is clearly above it.")

## 6 — Test, once, through `FilterFrameForVLM`

The five fold models are averaged, predictions are converted back to raw cosine units, and
**p4's algorithm** decides SEND/DISCARD at the τ that made the labels.

The per-quadrant table is the one to read: it is where `REFIXATION` — the case the whole
two-threshold design exists for — is either recovered or not.

In [ ]:
PRED = "/content/runs/joint/test_preds.csv"

!python -m gatetrain.infer \
    --folds_dir "{FOLDS}" --ckpt_dir "{RUNS}" --thresholds "{THRESH}" --out_csv "{PRED}"

### The test result, in one picture

In [ ]:
import sys; sys.path.insert(0, "/content/GazeVLM")
from src.foldtrain.metrics import roc_auc

pr = pd.read_csv(PRED)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))

# predicted vs true, for both scores
for k, (pc, tc, tau) in enumerate([("pred_S_frame", "frame_similarity", TAU_F),
                                   ("pred_S_gaze", "gaze_patch_token_sim", TAU_G)]):
    ax[k].scatter(pr[tc], pr[pc], s=4, alpha=.15)
    lo = min(pr[tc].min(), pr[pc].min()); hi = max(pr[tc].max(), pr[pc].max())
    ax[k].plot([lo, hi], [lo, hi], "r--", lw=1, label="perfect")
    ax[k].axvline(tau, c="k", lw=.8, ls=":"); ax[k].axhline(tau, c="k", lw=.8, ls=":")
    r = np.corrcoef(pr[tc], pr[pc])[0, 1]
    ax[k].set_xlabel(f"true {tc}"); ax[k].set_ylabel(f"predicted")
    ax[k].set_title(f"{tc}\nr = {r:+.3f}   (dotted = tau)"); ax[k].legend(fontsize=8)

cm = pd.crosstab(pr.true_quad, pr.pred_quad).reindex(
    index=list(NAMES.values()), columns=list(NAMES.values()), fill_value=0)
im = ax[2].imshow(cm.to_numpy(), cmap="Blues")
ax[2].set_xticks(range(4)); ax[2].set_xticklabels(NAMES.values(), rotation=30, ha="right")
ax[2].set_yticks(range(4)); ax[2].set_yticklabels(NAMES.values())
for i in range(4):
    for j in range(4):
        v = cm.to_numpy()[i, j]
        ax[2].text(j, i, f"{v:,}", ha="center", va="center", fontsize=8,
                   color="white" if v > cm.to_numpy().max()/2 else "black")
ax[2].set_xlabel("predicted"); ax[2].set_ylabel("truth"); ax[2].set_title("quadrants")
plt.tight_layout(); plt.show()

print("Points in the off-diagonal QUADRANTS of the left two panels are the errors that")
print("matter: the model put a row on the wrong side of tau, which flips the decision.")
print(f"\ntest accuracy on quadrants: {pr.correct.mean():.4f}")
display(pr.reason.value_counts().rename("rows").to_frame()
        .assign(pct=lambda d: (100*d.rows/len(pr)).round(1)))

---

## Reading it

| | |
|---|---|
| **R²** | read first. ≤ 0 means no better than predicting the mean, whatever AUC says |
| **AUC_gate** | ranking quality on `min(Ŝ_frame−τ_f, Ŝ_gaze−τ_g)` — the margin on the weaker axis |
| **REFIXATION recall** | the quadrant that justifies two thresholds. Near zero is a finding, not a detail |
| **FALSE SKIP** | content lost, unrecoverable. Not symmetric with FALSE SEND |
| **fold spread** | wide → the headline is mostly which videos landed in validation |
| **control** | must fail. If it does not, stop and fix the leak |

All CV figures are the **mean over the last 5 epochs**, never the best epoch — a maximum
over a noisy curve is biased upward even when there is no signal at all.

## If R² is at or below zero

1. Try `--rates_col gaze_vec3d_rates_window` — sphere-exact, where `ω_mag` over-reads by
   1/cos(pitch).
2. Try `--nce_weight 0.1` — a plain sum may be letting `L_NCE` swamp `L_sim`.
3. Compare against `--sim_only`. If that is better, the contrastive term is hurting.
4. Check the `--shuffle_control` landed at zero. If it did not, the problem is the
   evaluation, not the model.

## What is not tested here

Whether these thresholds correspond to anything a **VLM** would notice. The labels are
DINOv2 cosines cut at percentiles we chose. Until a VLM is run on gated vs ungated frames,
every number here is a proxy for the thing that actually matters.